# ICEEICT 2027 — Round 2: three backbones x three seeds, two datasets, budget sweep, per-character variants

Requires the Kaggle Dataset `bnuni-bundle-v2` (built by `tools/make_kaggle_bundle.py`; Kaggle unzips it) attached under Input, and a GPU (T4 x2 or P100). Internet on.

Runs: fine-tune xlm-roberta-base, google/muril-base-cased and csebuetnlp/banglabert on BD-SHS (3 seeds each, about 90 minutes total), then evaluate all 9 own models plus the 2 public checkpoints on the BD-SHS test split and on the BLP-2025 test split with families F1, F2, F2 per-character, NBSP control, F5 and budgets 1/3/5/10/length-aware. Output: `/kaggle/working/bnuni_round2_results.zip` (results and training logs only, no checkpoints).

In [ ]:
import glob, os, shutil, subprocess, sys, zipfile, time
PROJ = '/kaggle/working/proj'
shutil.rmtree(PROJ, ignore_errors=True); os.makedirs(PROJ)
zips = glob.glob('/kaggle/input/**/bnuni_bundle.zip', recursive=True)
trees = [os.path.dirname(os.path.dirname(os.path.dirname(p))) for p in glob.glob('/kaggle/input/**/src/eval/run_eval.py', recursive=True)]
if zips:
    zipfile.ZipFile(zips[0]).extractall(PROJ); print('extracted', zips[0])
elif trees:
    for item in ('src', 'data', 'fonts', 'SSOT.md'):
        s = os.path.join(trees[0], item)
        if os.path.isdir(s): shutil.copytree(s, os.path.join(PROJ, item))
        elif os.path.isfile(s): shutil.copy(s, PROJ)
    print('copied extracted bundle from', trees[0])
else:
    print('Contents of /kaggle/input:'); [print(' ', p) for p in glob.glob('/kaggle/input/**', recursive=True)[:60]]
    raise SystemExit('bundle not found: attach the bnuni-bundle-v2 dataset under Input')
os.chdir(PROJ)
assert os.path.exists('data/raw/blp25/test.csv'), 'this is the OLD bundle: upload dist/bnuni_bundle.zip from the project folder as a new dataset bnuni-bundle-v2'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentencepiece', 'protobuf'], check=False)
import torch, transformers; print('torch', torch.__version__, 'transformers', transformers.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# Fine-tune three backbones x three seeds on BD-SHS train (same hyperparameters for all)
BACKBONES = {'xlmr': 'xlm-roberta-base', 'muril': 'google/muril-base-cased', 'banglabert': 'csebuetnlp/banglabert'}
t0 = time.time()
for key, base in BACKBONES.items():
    for seed in (1, 2, 3):
        out = f'/kaggle/working/models/{key}_seed{seed}'
        if os.path.exists(out + '/train_log.json'):
            print('exists', out); continue
        subprocess.run([sys.executable, 'src/train/finetune_xlmr.py', '--base', base, '--seed', str(seed), '--out', out,
                        '--epochs', '3', '--lr', '2e-5', '--bs', '32', '--max-length', '128'], check=True)
        print(f'{key} seed {seed} done, {(time.time()-t0)/60:.0f} min elapsed')

In [ ]:
# Evaluation on both datasets. Own models + the two public checkpoints.
MODELS = [f'/kaggle/working/models/{k}_seed{s}:1' for k in ('xlmr', 'muril', 'banglabert') for s in (1, 2, 3)]
MODELS += ['Hate-speech-CNERG/bengali-abusive-MuRIL:1', 'tajuarAkash/enhanced-banglabert-hate-speech:1']
FAMS = ['F1', 'F2', 'F2nbsp', 'F5', 'F2_zwnj', 'F2_zwj', 'F2_zwsp', 'F2_wj', 'F2_shy']
common = ['--families', *FAMS, '--budgets', '1', '3', '5', '10', 'len', '--bs', '128']
for ds in ('bdshs', 'blp25'):
    subprocess.run([sys.executable, 'src/eval/run_eval.py', '--split', 'test', '--dataset', ds, '--out', f'/kaggle/working/results/{ds}_final', '--models', *MODELS, *common], check=True)
    subprocess.run([sys.executable, 'src/stats/analyze.py', '--dir', f'/kaggle/working/results/{ds}_final', '--split', 'test'], check=False)

In [ ]:
# Package results and training logs only (no checkpoints)
with zipfile.ZipFile('/kaggle/working/bnuni_round2_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk('/kaggle/working/results'):
        for f in files: z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), '/kaggle/working'))
    for p in glob.glob('/kaggle/working/models/*/train_log.json') + glob.glob('/kaggle/working/models/*/config.json'):
        z.write(p, os.path.relpath(p, '/kaggle/working'))
print('done: /kaggle/working/bnuni_round2_results.zip', os.path.getsize('/kaggle/working/bnuni_round2_results.zip') // 1024, 'KB')
print('Download ONLY this zip from the Output panel; the models/ folder is not needed.')